<a href="https://colab.research.google.com/github/eddy-udoh/fyp-yoruba-english-s2st/blob/main/notebooks/train_marian_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MedSpeak YO↔EN — MarianMT Fine-Tuning (Colab T4)

Trains **both** translation directions on a domain-balanced corpus (medical + general + greetings), with checkpoints written to Google Drive so a dropped session never costs more than one epoch.

**Before you run anything:** `Runtime → Change runtime type → Hardware accelerator: GPU (T4)`.

Run the cells top to bottom. If the session disconnects, just reconnect and re-run the training cell — it auto-resumes from the last Drive checkpoint.

## 1. Confirm the GPU

In [1]:
!nvidia-smi

Wed Jun 17 23:46:09 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Clone the repo
Pulls the latest code (including the domain-balanced training script) from GitHub.

In [2]:
!git clone https://github.com/eddy-udoh/fyp-yoruba-english-s2st.git
%cd fyp-yoruba-english-s2st
!git log --oneline -1

Cloning into 'fyp-yoruba-english-s2st'...
remote: Enumerating objects: 111, done.
remote: Counting objects: 100% (111/111), done.
remote: Compressing objects: 100% (86/86), done.
remote: Total 111 (delta 21), reused 110 (delta 20), pack-reused 0 (from 0)
Receiving objects: 100% (111/111), 3.28 MiB | 7.56 MiB/s, done.
Resolving deltas: 100% (21/21), done.
/content/fyp-yoruba-english-s2st
ec873a3 (HEAD -> main, origin/main, origin/HEAD) Add Colab notebook for bidirectional MarianMT fine-tuning


## 3. Install training dependencies
Only what training needs — deliberately NOT `requirements.txt` (which pulls Coqui `TTS`/`librosa` that fight Colab's NumPy and aren't used here). `torch` ships with Colab.

In [3]:
!pip install -q -U transformers datasets sacrebleu sentencepiece accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 116.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 42.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 37.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 14.9 MB/s eta 0:00:00


## 4. Mount Google Drive (checkpoint persistence)
Checkpoints + the final model are written straight to Drive, so they survive a disconnect and you keep them after the session ends.

In [4]:
from google.colab import drive
drive.mount('/content/drive')

YOEN_DIR = '/content/drive/MyDrive/marian-yoruba-medical'
ENYO_DIR = '/content/drive/MyDrive/marian-english-yoruba'
print('yo->en checkpoints ->', YOEN_DIR)
print('en->yo checkpoints ->', ENYO_DIR)

Mounted at /content/drive
yo->en checkpoints -> /content/drive/MyDrive/marian-yoruba-medical
en->yo checkpoints -> /content/drive/MyDrive/marian-english-yoruba


## 5. Train  yo → en  (Yorùbá → English)
~20–40 min on a T4. Watch the per-epoch **eval BLEU** and the **greeting sanity check** printed at the end.

If the session dropped mid-run, just re-run this cell — you'll see `[RESUME] Found checkpoint …`.

In [5]:
!python src/nmt/marian_finetune.py --direction yo-en --output-dir "$YOEN_DIR"


  FYP S2ST — MarianMT Fine-Tune [yo-en] | Phase 3 Step 2
  Direction : yo-en   (base: Helsinki-NLP/opus-mt-yo-en)
  Device    : GPU — Tesla T4
  Effective batch : 16  (per_device=8 × grad_accum=2)

  1 / 4  Loading & blending data
  Medical train: 12000 rows
  Medical val: 1500 rows
  Medical test (held out): 1500 rows

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'opus100' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
README.md: 100% 65.4k/65.4k [00:00<00:00, 121MB/s]
  [WARN] Could not load opus100 general data (Invalid HF URI 'hf://datasets/opus100@805090dc28bf78897da9641cdf08b61287580df9/.huggingface.yaml'. Repository id must be 'namespace/name', got 'opus100'.).
  [WARN] Proceeding with medical + greetings only.
  General train  : 0 rows
  Greetings train: 900 rows (18 unique × 50)



## 6. Train  en → yo  (English → Yorùbá)
Same again for the other direction. The script auto-prefixes `>>yor<< ` and uses `opus-mt-en-nic` as the base.

In [6]:
!python src/nmt/marian_finetune.py --direction en-yo --output-dir "$ENYO_DIR"


  FYP S2ST — MarianMT Fine-Tune [en-yo] | Phase 3 Step 2
  Direction : en-yo   (base: Helsinki-NLP/opus-mt-en-nic)
  Device    : GPU — Tesla T4
  Effective batch : 16  (per_device=8 × grad_accum=2)

  1 / 4  Loading & blending data
  Medical train: 12000 rows
  Medical val: 1500 rows
  Medical test (held out): 1500 rows

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'opus100' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
  [WARN] Could not load opus100 general data (Invalid HF URI 'hf://datasets/opus100@805090dc28bf78897da9641cdf08b61287580df9/.huggingface.yaml'. Repository id must be 'namespace/name', got 'opus100'.).
  [WARN] Proceeding with medical + greetings only.
  General train  : 0 rows
  Greetings train: 900 rows (18 unique × 50)

  Blended TRAIN  : 12900 rows total
    medical   

## 7. Save eval results to Drive & verify the models
The model weights are already on Drive (via `--output-dir`). This copies the per-direction eval JSONs too, then lists what was produced.

In [7]:
!mkdir -p /content/drive/MyDrive/fyp_eval
!cp evaluation/nmt_finetuned_*.json /content/drive/MyDrive/fyp_eval/ 2>/dev/null; echo 'eval JSONs copied'
print('\n--- yo->en model files ---')
!ls -lh "$YOEN_DIR" | grep -E 'safetensors|config.json|spm|tokenizer'
print('\n--- en->yo model files ---')
!ls -lh "$ENYO_DIR" | grep -E 'safetensors|config.json|spm|tokenizer'

eval JSONs copied

--- yo->en model files ---

--- en->yo model files ---


## 8. Integrate back into the repo (do this on your local machine)

The trained weights live on your Drive. The API ([`src/api/app.py`](../src/api/app.py)) loads each model from the **top-level** folder, so drop them straight in:

```
models/marian-yoruba-medical/      ← contents of Drive's marian-yoruba-medical/
models/marian-english-yoruba/      ← contents of Drive's marian-english-yoruba/
```

Steps locally:
1. Download both folders from Google Drive (`marian-yoruba-medical`, `marian-english-yoruba`).
2. Copy their **contents** (config.json, *.spm, tokenizer files, **model.safetensors**) into the matching `models/<dir>/` folders in the repo.
3. Each folder must contain `config.json`, `generation_config.json`, `source.spm`, `target.spm`, `tokenizer_config.json`, `vocab.json`, and `model.safetensors`.
4. Start the API — it logs `Loading fine-tuned en→yo model …` when it finds them.

> The weights are git-ignored (too large for GitHub), so they stay out of version control — keep the Drive copy as your backup.

**Then run the evaluation** (`src/eval/end_to_end_eval.py`, `src/eval/results_report.py`) on the machine that has the models + GPU.